In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [13]:
df = pd.read_csv('combined_crypto_data.csv')

In [14]:
df.sample(5)

,Date,Open,High,Low,Close,Volume,coin
27944,2019-10-10,193.191076,193.896929,188.306469,191.659672,8.375913e+09,Ethereum
29091,2019-02-07,0.025558,0.026112,0.025508,0.025702,1.556385e+08,Tron
31940,2019-12-26,0.043572,0.045710,0.043512,0.045101,1.598656e+08,Stellar
36069,2021-03-04,0.050587,0.051087,0.047879,0.050048,1.170617e+09,Dogecoin
28272,2020-09-02,477.007884,480.330986,424.460019,440.040507,1.969185e+10,Ethereum


In [15]:
df.shape

(37082, 7)

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37082 entries, 0 to 37081
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    37082 non-null  object 
 1   Open    37082 non-null  float64
 2   High    37082 non-null  float64
 3   Low     37082 non-null  float64
 4   Close   37082 non-null  float64
 5   Volume  37082 non-null  float64
 6   coin    37082 non-null  object 
dtypes: float64(5), object(2)
memory usage: 2.0+ MB


In [17]:
df.describe()

,Open,High,Low,Close,Volume
count,37082.000000,37082.000000,37082.000000,37082.000000,3.708200e+04
mean,985.323755,1016.058015,952.987707,987.120511,3.022542e+09
std,5088.101367,5249.503670,4907.932082,5093.703878,1.190963e+10
min,0.000086,0.000089,0.000079,0.000086,0.000000e+00
25%,0.072456,0.075664,0.069536,0.072648,4.937190e+06
50%,1.001157,1.008733,0.999850,1.001138,8.512805e+07
75%,30.459673,31.916399,28.996246,30.512205,9.388489e+08
max,63523.754869,64863.098908,62208.964366,63503.457930,3.509679e+11


In [18]:
df.isnull().sum()

Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
coin      0
dtype: int64

In [19]:
df.duplicated().sum()

np.int64(0)

In [20]:
# Converting to datetime datatype
df['Date'] = pd.to_datetime(df['Date'])

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37082 entries, 0 to 37081
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    37082 non-null  datetime64[ns]
 1   Open    37082 non-null  float64       
 2   High    37082 non-null  float64       
 3   Low     37082 non-null  float64       
 4   Close   37082 non-null  float64       
 5   Volume  37082 non-null  float64       
 6   coin    37082 non-null  object        
dtypes: datetime64[ns](1), float64(5), object(1)
memory usage: 2.0+ MB


In [22]:
df['coin'].value_counts()

coin
Bitcoin           2991
Litecoin          2991
XRP               2893
Dogecoin          2760
Monero            2602
Stellar           2527
Tether            2318
NEM               2288
Ethereum          2160
Iota              1484
EOS               1466
BinanceCoin       1442
Tron              1392
ChainLink         1385
Cardano           1374
USDCoin           1002
CryptocomCoin      935
WrappedBitcoin     888
Cosmos             845
Solana             452
Polkadot           320
Uniswap            292
Aave               275
Name: count, dtype: int64

In [23]:
def create_features(df):

    df = df.sort_values(by=['coin', 'Date'])

    grouped = df.groupby('coin')

    df['return_1'] = grouped['Close'].pct_change(1)
    df['return_3'] = grouped['Close'].pct_change(3)

    df['ma_7'] = grouped['Close'].rolling(7).mean().reset_index(level=0, drop=True)
    df['ma_14'] = grouped['Close'].rolling(14).mean().reset_index(level=0, drop=True)

    df['ma_distance_7'] = (df['Close'] - df['ma_7']) / df['ma_7']

    df['volatility_7'] = grouped['return_1'].rolling(7).std().reset_index(level=0, drop=True)

    df['volume_change'] = grouped['Volume'].pct_change(1)

    df['ma_crossover'] = df['ma_7'] - df['ma_14']

    df['target'] = grouped['Close'].shift(-1) > df['Close']
    df['target'] = df['target'].astype(int)

    df = df.dropna()

    return df

In [24]:
df = create_features(df)

In [25]:
counts = df['coin'].value_counts()

def group_coin(coin):
    if counts[coin] < 1300:
        return "Other"
    else:
        return coin

df['coin_grouped'] = df['coin'].apply(group_coin)

In [26]:
coins_encoded = pd.get_dummies(df['coin_grouped']).astype(int)

In [27]:
features = [
    'return_1',
    'return_3',
    'ma_distance_7',
    'volatility_7',
    'volume_change',
    'ma_crossover',
    'target'
]


In [28]:
df_train = pd.concat([df[features], coins_encoded], axis=1)

In [29]:
X = df_train.drop('target', axis=1)
y = df_train['target']

In [30]:
num_features = ['return_1', 'return_3','ma_distance_7', 'volatility_7','volume_change','ma_crossover']

In [36]:
df.loc[:, features] = df.loc[:, features].clip(-10, 10)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


models = {}
summary = []

coins = df["coin"].unique()

for coin in coins:
    print(f"\nTraining model for {coin}")

    coin_df = df[df["coin"] == coin].copy()

    if len(coin_df) < 50:
        print("Skipped (not enough data)")
        continue

    X = coin_df[features]
    y = coin_df["target"]

    split = int(len(X) * 0.8)

    X_train = X.iloc[:split]
    X_test = X.iloc[split:]

    y_train = y.iloc[:split]
    y_test = y.iloc[split:]

    scaler = RobustScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)

    # ================================
    # METRICS
    # ================================
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    print(f"Accuracy  : {acc:.3f}")
    print(f"Precision : {prec:.3f}")
    print(f"Recall    : {rec:.3f}")
    print(f"F1 Score  : {f1:.3f}")
    print(f"Confusion Matrix:\n{cm}")

    summary.append({
        "coin": coin,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    })

    # ================================
    # SAVE MODEL DATA
    # ================================
    models[coin] = {
        "weights": model.coef_[0].tolist(),
        "bias": float(model.intercept_[0]),
        "scaler_center": scaler.center_.tolist(),
        "scaler_scale": scaler.scale_.tolist(),
        "metrics": {
            "accuracy": float(acc),
            "precision": float(prec),
            "recall": float(rec),
            "f1_score": float(f1),
            "confusion_matrix": cm.tolist()
        },
        "features": features
    }

# ================================
# SAVE JSON FILE
# ================================
with open("models.json", "w") as f:
    json.dump(models, f, indent=2)

print("\n✅ All models saved to models.json")

# ================================
# SUMMARY TABLE
# ================================
summary_df = pd.DataFrame(summary)

print("\n📊 Model Performance Summary:")
print(summary_df.sort_values("accuracy", ascending=False))


Training model for Aave
Accuracy  : 1.000
Precision : 1.000
Recall    : 1.000
F1 Score  : 1.000
Confusion Matrix:
[[26  0]
 [ 0 27]]

Training model for BinanceCoin
Accuracy  : 1.000
Precision : 1.000
Recall    : 1.000
F1 Score  : 1.000
Confusion Matrix:
[[129   0]
 [  0 157]]

Training model for Bitcoin
Accuracy  : 1.000
Precision : 1.000
Recall    : 1.000
F1 Score  : 1.000
Confusion Matrix:
[[252   0]
 [  0 298]]

Training model for Cardano
Accuracy  : 1.000
Precision : 1.000
Recall    : 1.000
F1 Score  : 1.000
Confusion Matrix:
[[125   0]
 [  0 148]]

Training model for ChainLink
Accuracy  : 1.000
Precision : 1.000
Recall    : 1.000
F1 Score  : 1.000
Confusion Matrix:
[[128   0]
 [  0 147]]

Training model for Cosmos
Accuracy  : 1.000
Precision : 1.000
Recall    : 1.000
F1 Score  : 1.000
Confusion Matrix:
[[84  0]
 [ 0 83]]

Training model for CryptocomCoin
Accuracy  : 1.000
Precision : 1.000
Recall    : 1.000
F1 Score  : 1.000
Confusion Matrix:
[[ 84   0]
 [  0 101]]

Training mod